In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sqlite3
import os
import sys
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname("star_functions.py"), '..')))
import star_functions as nana
import numpy as np
import matplotlib.pyplot as plt
import lightkurve as lk
from astropy.io import fits
from astropy.table import Table

In [ ]:
def get_db_connection():

    conn = sqlite3.connect('/Users/nana/venv/hoggnation/oscillator_catalog/2026-03-19modes.db', timeout=120.0)

    conn.execute("PRAGMA journal_mode=WAL")
    return conn

In [ ]:
def fit_fourier_series_to_good_parents(mid,M):
    '''
    # fit_fourier_series_to_good_parents()
    Fit Fourier series to all good parent modes

    ## Inputs:
    - `M`: number of harmonics to include in the fit

    ## Comment:
    - This function is a placeholder and needs to be implemented based on the specific requirements of the Fourier fitting process for multiple modes.
    '''
    #query parnet mode view of db to get all parents with variance > 1e-6
    conn = get_db_connection()
    cursor = conn.cursor()
    #cursor.execute("SELECT mode_id, parent_frequency, star_id, num_of_children FROM parent_modes WHERE variance > 1e-4")
    cursor.execute(f"SELECT mode_id, parent_frequency, star_id, num_of_children FROM parent_modes WHERE mode_id = {mid}")

    rows = cursor.fetchall()

    #initialize an astropy table, the features will have length 2M + 1
    output_table = Table()
    parent_mode_ids = []
    star_ids = []
    orig_freqs = []
    refined_freqs = []
    num_children = []
    constants = []
    amp_as = [[] for m in range(1, M+1)]  # one list per harmonic
    amp_bs = [[] for m in range(1, M+1)]

    #for each parent, call fit_fourier_series_to_mode() and aggregate results
    for parent_mode_id, orig_freq, star_id, num_child in rows:
        try: 
            new_freq, features = nana.fit_fourier_series_to_mode(parent_mode_id, M)
            print(f"star {star_id}: original freq {orig_freq}, {orig_freq - new_freq} difference")
            print("REFINED FREQUENCY", new_freq, "M", M)
            #add new_freq and features to the output table
            parent_mode_ids.append(parent_mode_id)
            star_ids.append(star_id)
            orig_freqs.append(orig_freq)
            refined_freqs.append(new_freq)
            num_children.append(num_child)
            constants.append(features[0])
            for m in range(1, M+1):
                amp_as[m-1].append(features[2*m-1])
                amp_bs[m-1].append(features[2*m])
        except Exception as e:
            print(f"Skipping {star_id} mode {parent_mode_id}: {str(e)}")
            continue
    
    output_table = Table()
    output_table['parent_mode_id'] = parent_mode_ids
    output_table['star_id'] = star_ids
    output_table['original_frequency'] = orig_freqs
    output_table['refined_frequency'] = refined_freqs
    output_table['num_children'] = num_children
    output_table['constant'] = constants
    for m in range(1, M+1):
        output_table[f'amplitude_a{m}'] = amp_as[m-1]
        output_table[f'amplitude_b{m}'] = amp_bs[m-1]
     

In [ ]:
newfreq, features = nana.fit_fourier_series_to_mode(83645, 32)
print(features)
print(len(features))

In [ ]:
table = Table.read('good_parents_series_fit.fits')



In [ ]:
#compute the relevant sums for plotting
n = 2
sums = np.zeros(32)
parent_modes = np.zeros(1519)
j = 0
for i in range(len(features)-1):
    if i%2 != 0:
        sums[j] = (features[i]**2 + features[i+1]**2)/ (features[0]**2)
        j = j+1
        

amplitude_cols = ['constant'] + [f'amplitude_a{i}' for i in range(1, 33)] + [f'amplitude_b{i}' for i in range(1, 33)]
features = np.array(list(table[0][amplitude_cols]))  # 1D array of length 64

sums = np.zeros(32)
j = 0
for i in range(len(features)-1):
    if i % 2 != 0:
        sums[j] = (features[i]**2 + features[i+1]**2) / (features[0]**2)
        j += 1


In [ ]:
features_all = np.ma.filled(
    np.array([table[col].data for col in amplitude_cols]).T,
    fill_value=np.nan
)  # shape (n_rows, 64)

all_sums = []
for features in features_all:
    if np.any(np.isnan(features)):
        continue
    sums = np.zeros(32)
    j = 0
    for i in range(len(features)-1):
        if i % 2 != 0:
            sums[j] = (features[i]**2 + features[i+1]**2) / (features[0]**2)
            j += 1
    all_sums.append(sums)

all_sums = np.array(all_sums)

# Plot the three pairs
pairs = [(0,1), (0,2), (1,2)]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (i, j) in zip(axes, pairs):
    ax.scatter(all_sums[:, i], all_sums[:, j], s=5, alpha=0.4)
    ax.set_xlabel(f'sum{i+1}')
    ax.set_ylabel(f'sum{j+1}')
    ax.set_title(f'sum{i+1} vs sum{j+1}')
    ?

plt.tight_layout()
plt.show()



In [ ]:
def fourier_fitting_parent_mode(parent_mode_id, M = 1):
    conn = get_db_connection()
    cursor = conn.cursor()

    # get star_id and frequency from db
    cursor.execute(f"SELECT star_id, frequency FROM mode WHERE mode_id = {parent_mode_id}")
    star_id, new_freq = cursor.fetchone()

    # get children
    cursor.execute(f"""SELECT mode_id, frequency FROM mode 
                   WHERE parent_mode_id = {int(parent_mode_id)}""")
    children = cursor.fetchall()
    num_children = len(children)
    conn.close()

    # get lightcurve
    lc, delta_f, sampling_time, exptime = nana.get_kepler_data(star_id)
    t_fit, flux_fit, weight_fit = nana.mask_vals(lc, star_id)
    
    # plt.plot(lc.time.value, lc.flux.value, '.', markersize=0.5, color='black') 
    # plt.title(f"Lightcurve of {star_id}")
    # plt.xlabel("Time (days)")
    # plt.ylabel("Flux")
    # plt.show()

    over_sampling = 3
    df, f_maxC = delta_f/over_sampling, (3 / (2*sampling_time))
    f_min = over_sampling * df
    
    freq_full, power_full = nana.get_periodogram(f_min, f_maxC, df, lc, star_id)
    fc = nana.folding_freq(delta_f, freq_full, powaer_full, sampling_time, star_id, False)

    conn = nana.get_db_connection()
    cursor = conn.cursor()
    cursor.execute(f"SELECT parent_frequency FROM parent_modes WHERE star_id = '{star_id}'")
    parent_freqs = [row[0] for row in cursor.fetchall()]
    cursor.execute(f"SELECT frequency FROM mode WHERE star_id = '{star_id}' AND parent_mode_id IS NOT NULL")
    child_freqs = [row[0] for row in cursor.fetchall()]
    conn.close()

    # plt.plot(freq_full, power_full, 'k.', markersize=1, alpha=0.5)
    # plt.axvline(fc)
    # plt.axvline(fc/2)
    # plt.xlabel("Frequency (1/day)")
    # plt.ylabel("Power")
    # plt.semilogy()
    # plt.title(f"Full Log Periodogram of {star_id}")
    # for freq in parent_freqs:
    #     plt.axvline(freq, color='red', alpha=0.6, lw=1)
    # for freq in child_freqs:
    #     plt.axvline(freq, color='red', alpha=0.2, lw=0.5)
    # plt.show()

    print(f"parent {parent_mode_id} (freq: {new_freq:.4f}) has {num_children} children")

    # fit parent
    new_freq, parent_features = nana.fit_fourier_series_to_mode(parent_mode_id, M)

    theta = np.linspace(0, 4*np.pi, 1000)
    yplot0 = np.zeros_like(theta)

    # add parent harmonics
    for m in range(1, M+1):
        a = parent_features[2*m-1]
        b = parent_features[2*m]
        yplot0 += a * np.cos(m * theta) + b * np.sin(m * theta)

    # add child harmonics
    # for child in children:
    #     child_mode_id = child[0]
    #     nu_child = child[1]
    #     conn = nana.get_db_connection()
    #     cursor = conn.cursor()
    #     cursor.execute(f"SELECT amplitude_a, amplitude_b FROM mode WHERE mode_id = {child_mode_id}")
    #     a, b = cursor.fetchone()
    #     conn.close()
    #     if a is not None and b is not None:
    #         yplot0 += a * np.cos(theta * nu_child / new_freq) \
    #                 + b * np.sin(theta * nu_child / new_freq)

    omega = 2 * np.pi * new_freq
    phase = (omega * t_fit) % (4 * np.pi)
    yplot = flux_fit - np.nanmean(flux_fit)

    plt.plot(phase, yplot, alpha=0.1, markersize=2, marker='.', linestyle='none', color='k')
    plt.plot(theta, yplot0, color='r', alpha=0.6, zorder=3)
    plt.xlabel('Phase')
    plt.ylabel('Flux')
    plt.title(f'folded ν: {new_freq:.4f} for {star_id} ({num_children} children), M = {m}')
    plt.show()

In [ ]:
numbers = np.array([1,2,4,8,16,32, 64])
for m in numbers:
    fourier_fitting_parent_mode(162626, m)

In [ ]:
numbers = np.array([1,2,4,8,16,32, 64])
for m in numbers:
    fourier_fitting_parent_mode(206552, m)

In [ ]:
numbers = np.array([1,2,4,8,16,32])
for m in numbers:
    fourier_fitting_parent_mode(32349, m)

In [ ]:
numbers = np.array([1,2,4,8,16,32])
for m in numbers:
    fourier_fitting_parent_mode(111904, m)


In [ ]:
numbers = np.array([2,4,8,16,32])
for m in numbers:
    fourier_fitting_parent_mode(31122, m)


In [ ]:
nana.fit_fourier_series_to_good_parents(32)

In [ ]:
from astropy.table import Table
t = Table.read('good_parents.fits')
print(t)

In [ ]:
from astropy.table import Table
t = Table.read('good_parents.fits')
t['freq_deviation'] = t['refined_frequency'] - t['original_frequency']
print(t['parent_mode_id', 'star_id', 'original_frequency', 'refined_frequency', 'freq_deviation'])

In [ ]:
newfreq, features = nana.fit_fourier_series_to_mode(83645, 2)
print("new freq", newfreq)
print("features", features)

In [ ]:
fourier_fitting_parent_mode(83645, 2)

In [ ]:
nana.find_modes_in_star("KIC011724091", M = 1)

In [ ]:
result = nana.find_modes_in_star("KIC006859813", M = 4)
nana.output_modes_to_db('KIC006859813', 'Kepler_long', result, M=4)

In [ ]:
result = nana.find_modes_in_star("KIC012257908", M = 4)
nana.output_modes_to_db('KIC012257908', 'Kepler_long', result, M=4)

In [ ]:
conn

In [ ]:
#select stars with sqlite3 query

def parent_view_srt():
    conn = get_db_connection()
    cursor = conn.cursor()
    
    cursor.execute("SELECT DISTINCT star_id FROM mode")
    stars = cursor.fetchall()
    
    
    for star in stars:
        
        cursor.execute(f"SELECT frequency, mode_id FROM mode WHERE star_id = '{star[0]}'") 
        results = np.array(cursor.fetchall())
        modes = results[:,0]
        mode_ids = results[:,1]
    
        ind = np.argsort(modes)
        modes = modes[ind]
        mode_ids = mode_ids[ind]
        parents = np.zeros_like(modes).astype(int)-1
        
        for i, mode in enumerate(modes):
            if parents[i] < 0:
                for j in range(i+1, len(modes)):
                    q = np.round(modes[j]/modes[i]) 
                    if np.abs(modes[j] -  q*modes[i]) < 1/1470: #this is dif for each star
                        parents[j] = mode_ids[i]
                        cursor.execute(f"UPDATE mode SET parent_mode_id = {int(parents[j])} WHERE mode_id = {int(mode_ids[j])}")
    
        conn.commit()
        
    conn.close()
#bug do we need to close after ALL commits

In [ ]:
def harmonic_visualization_multiple(star_id):
    
    conn = nana.get_db_connection()
    cursor = conn.cursor()
    
    lc, delta_f, sampling_time, exptime = nana.get_kepler_data(star_id)
    t_fit, flux_fit, weight_fit = nana.mask_vals(lc, star_id)
    
    plt.plot(lc.time.value, lc.flux.value, '.', markersize=0.5, color='black') 
    plt.title(f"Lightcurve of {star_id}")
    plt.xlabel("Time (days)")
    plt.ylabel("Flux")
    plt.show()

    over_sampling = 3
    df, f_maxC = delta_f/over_sampling, (3 / (2*sampling_time))
    f_min = over_sampling * df
    
    freq_full, power_full = nana.get_periodogram(f_min, f_maxC, df, lc, star_id)
    fc = nana.folding_freq(delta_f, freq_full, power_full, sampling_time, star_id, False)

    cursor.execute(f"SELECT parent_frequency FROM parent_modes WHERE star_id = '{star_id}'")
    parent_freqs = [row[0] for row in cursor.fetchall()]
    
    cursor.execute(f"SELECT frequency FROM mode WHERE star_id = '{star_id}' AND parent_mode_id IS NOT NULL")
    child_freqs = [row[0] for row in cursor.fetchall()]
    
    plt.plot(freq_full, power_full, 'k.', markersize=1, alpha=0.5)
    plt.axvline(fc)
    plt.axvline(fc/2)
    plt.xlabel("Frequency (1/day)")
    plt.ylabel("Power")
    plt.semilogy()
    plt.title(f"Full Log Periodogram of {star_id}")
    for freq in parent_freqs:
        plt.axvline(freq, color='red', alpha=0.6, lw=1)
    for freq in child_freqs:
        plt.axvline(freq, color='red', alpha=0.2, lw=0.5)
    plt.show()

    cursor.execute(f"SELECT COUNT(*) FROM mode WHERE star_id = '{star_id}'")
    total_modes = cursor.fetchone()[0]

    cursor.execute(f"""SELECT mode_id, frequency FROM mode 
                   WHERE star_id = '{star_id}' AND parent_mode_id IS NULL""")
    parents = cursor.fetchall()

    print(f"{star_id} has {total_modes} modes and {len(parents)} parents:")

    for parent in parents:
        parent_mode_id = parent[0]
        nu_parent = parent[1]

        cursor.execute(f"""SELECT harmonic, amplitude_a, amplitude_b FROM amplitude 
                       WHERE mode_id = {parent_mode_id} ORDER BY harmonic""")
        parent_amps = {row[0]: (row[1], row[2]) for row in cursor.fetchall()}

        cursor.execute(f"""SELECT mode_id, frequency FROM mode 
                       WHERE parent_mode_id = {int(parent_mode_id)}""")
        children = cursor.fetchall()

        num_children = len(children)
        print(f"  parent {int(parent_mode_id)} (freq: {nu_parent:.4f}) has {num_children} children")

        theta = np.linspace(0, 4*np.pi, 1000)
        yplot0 = np.zeros_like(theta)

        for m, (a, b) in parent_amps.items():
            if a is not None and b is not None:
                yplot0 += a * np.cos(m * theta) + b * np.sin(m * theta)

        for child in children:
            child_mode_id = child[0]
            nu_child = child[1]
            cursor.execute(f"""SELECT harmonic, amplitude_a, amplitude_b FROM amplitude 
                           WHERE mode_id = {child_mode_id} ORDER BY harmonic""")
            child_amps = {row[0]: (row[1], row[2]) for row in cursor.fetchall()}
            for m, (a, b) in child_amps.items():
                if a is not None and b is not None:
                    yplot0 += a * np.cos(m * theta * nu_child / nu_parent) \
                            + b * np.sin(m * theta * nu_child / nu_parent)

        omega = 2 * np.pi * nu_parent
        phase = (omega * t_fit) % (4 * np.pi)
        yplot = flux_fit - np.nanmean(flux_fit)

        plt.plot(phase, yplot, alpha=0.1, markersize=2, marker='.', linestyle='none', color='k')
        plt.plot(theta, yplot0, color='r', alpha=0.6, zorder=3)
        plt.xlabel('Phase')
        plt.ylabel('Flux')
        plt.title(f'folded ν: {nu_parent:.4f} for {star_id} ({num_children} children)')
        plt.show()

    conn.close()

### M = 1, KIC011724091

In [ ]:
delete_data()
result = nana.find_modes_in_star("KIC011724091")
nana.output_modes_to_db('KIC011724091', 'Kepler_long', result, M=1)
parent_view_srt()
harmonic_visualization_multiple("KIC011724091")

### M = 5, KIC011724091

In [ ]:
delete_data()
result = nana.find_modes_in_star("KIC011724091", M = 5)
nana.output_modes_to_db('KIC011724091', 'Kepler_long', result, M=5)
parent_view_srt()
harmonic_visualization_multiple("KIC011724091")

### M = 10

In [ ]:
delete_data()
result = nana.find_modes_in_star("KIC011724091", M = 10)
nana.output_modes_to_db('KIC011724091', 'Kepler_long', result, M=10)
parent_view_srt()
harmonic_visualization_multiple("KIC011724091")

### M = 20

In [ ]:
delete_data()
result = nana.find_modes_in_star("KIC011724091", M = 20)
nana.output_modes_to_db('KIC011724091', 'Kepler_long', result, M=20)
parent_view_srt()
harmonic_visualization_multiple("KIC011724091")

### M = 40

In [ ]:
delete_data()
result = nana.find_modes_in_star("KIC011724091", M = 40)
nana.output_modes_to_db('KIC011724091', 'Kepler_long', result, M=40)
parent_view_srt()
harmonic_visualization_multiple("KIC011724091")

In [ ]:
delete_data()
result = nana.find_modes_in_star("KIC010858720", M = 1)
nana.output_modes_to_db('KIC010858720', 'Kepler_long', result, M=1)
parent_view_srt()
harmonic_visualization_multiple("KIC010858720")

In [ ]:
delete_data()
result = nana.find_modes_in_star("KIC010858720", M = 5)
nana.output_modes_to_db('KIC010858720', 'Kepler_long', result, M=5)
parent_view_srt()
harmonic_visualization_multiple("KIC010858720")

In [ ]:
delete_data()
result = nana.find_modes_in_star("KIC010858720", M = 10)
nana.output_modes_to_db('KIC010858720', 'Kepler_long', result, M=10)
parent_view_srt()
harmonic_visualization_multiple("KIC010858720")

In [ ]:
delete_data()
result = nana.find_modes_in_star("KIC010858720", M = 20)
nana.output_modes_to_db('KIC010858720', 'Kepler_long', result, M=20)
parent_view_srt()
harmonic_visualization_multiple("KIC010858720")

In [ ]:
delete_data()
result = nana.find_modes_in_star("KIC010858720", M = 40)
nana.output_modes_to_db('KIC010858720', 'Kepler_long', result, M=40)
parent_view_srt()
harmonic_visualization_multiple("KIC010858720")

In [ ]:
numbers = np.array([1,2,4,8,16,32, 64])
for m in numbers:
    fourier_fitting_parent_mode(255704, m)


    